In [1]:
# procedural memory 用于告诉模型在遇见某一类场景时该如何做出回应

In [2]:
profile = {
    "name": "jiarui",
    "full_name": "raojiarui",
    "user_profile_background": "Senior software engineer and also a university student"
}
prompt_instructions = {
    "triage_rules": {
        # 分类规则
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage emails.",
}
# 创建示例邮件
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
    Hi John,

    I was reviewing the API documentation for the new authentication service and noticed a few endpoints are missing. Specifically, I'm looking at:

    - /auth/refresh
    - /auth/validate

    Thanks!
    Alice
    """
}

# 定义长期记忆存储
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(
    index={"embed": "ollama:qwen3-embedding:0.6b"}
)

email = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}
data = {
    "email": email,
    "label": "respond"
}
import uuid

store.put(
    # 将其存入长期记忆
    ("email_assistant", "raojiarui", "examples"),
    str(uuid.uuid4()),
    data
)
template = """
Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content:
```
{content}
```
> Triage Result:{result}
"""


def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            template.format(
                subject=eg.value["email"]["subject"],
                to_email=eg.value["email"]["to"],
                from_email=eg.value["email"]["author"],
                content=eg.value["email"]["email_thread"][:400],
                result=eg.value["label"],
            )
        )
    return "\n\n--------------------\n\n".join(strs)


email_data = {
    "author": "Sarah Chen <sarah.chen@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Update: Backend API Changes Deployed to Staging",
    "email_thread": """Hi John,

wanted to let you know that I've deployed the new authentication endpoints we discussed to the staging environment. Key changes include:

- Implemented JWT refresh token rotation
- Added rate limiting for login attempts
- Updated API documentation with new endpoints

All tests are passing and the changes are ready for review. You can test it out at staging-api.company.com/auth/*

No immediate action needed from your side - just keeping you in the loop since this affects the systems you're working on.

Best regards,
Sarah
""",
}

# 查询对应邮件操作
results = store.search(
    ("email_assistant", "raojiarui", "examples"),
    query=str({"email": email_data}),  # 查询参数
    limit=1
)
# 将查询得到的结果通过函数进行format
print(format_few_shot_examples(results))
triage_system_prompt = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Background >
{user_profile_background}.
</ Background >

< Instructions >

{name} gets lots of emails. Your job is to categorize each email into one of three categories:

1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that {name} should know about but doesn't require a response
3. RESPOND - Emails that need a direct response from {name}

Classify the below email into one of these categories.

</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

There are also other things that {name} should know about, but don't require an email response. For these, you should notify {name} (using the `notify` response). Examples of this include:
{triage_notify}

Emails that are worth responding to:
{triage_email}
</ Rules >

< Few shot examples >

Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above

{examples}
</ Few shot examples >
"""

Here are some previous examples:

--------------------


Email Subject: Quick question about API documentation
Email From: Alice Smith <alice.smith@company.com>
Email To: John Doe <john.doe@company.com>
Email Content:
```
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice
```
> Triage Result:respond



In [3]:
# 创建路由规则和创建路由
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

llm = init_chat_model("deepseek-v4-flash", extra_body={"thinking": {"type": "disabled"}})


class Router(BaseModel):
    """Analyze the unread email and route it according to its content"""

    reasoning: str = Field(
        # 语言模型在此生成决策
        description="Step-by-Step reasoning behind the classification."
    )

    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: ignore for irrelevant emails,notify for important information that doesn't need a response,respond for emails that need a reply",
    )


llm_router = llm.with_structured_output(Router)  # 输出规则
from prompts import triage_user_prompt
from langgraph.graph import add_messages


class State(TypedDict):
    email_input: dict  # email输入
    messages: Annotated[list, add_messages]


from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display


# 构建代理
def triage_router(state: State, config, store) -> Command[
    Literal["response_agent", "__end__"]
]:
    # load the info
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    # 添加示例：想要通过embed模型进行存储，需要添加对应存储的namespace
    namespace = (
        "email_assistant",
        config['configurable']['langgraph_user_id'],
        "examples"
    )

    examples = store.search(
        namespace,
        # 查询语句
        query=str({"email": state["email_input"]})
    )

    examples = format_few_shot_examples(examples)

    # 获得user_id
    langgraph_user_id = config['configurable']['langgraph_user_id']
    namespace = (langgraph_user_id,)

    result = store.get(namespace, "triage_ignore")
    if result is None:
        store.put(
            namespace,
            "triage_ignore",
            {
                "prompt": prompt_instructions["triage_rules"]["ignore"]
            }
        )
        ignore_prompt = prompt_instructions["triage_rules"]["ignore"]
    else:
        ignore_prompt = result.value["prompt"]

    # 注意：store 中的 key 必须是 triage_notify（与官方一致），后面 prompts 列表要用同一个 key 取
    result = store.get(namespace, "triage_notify")
    if result is None:
        store.put(
            namespace,
            "triage_notify",
            {
                "prompt": prompt_instructions["triage_rules"]["notify"]
            }
        )
        notify_prompt = prompt_instructions["triage_rules"]["notify"]
    else:
        notify_prompt = result.value["prompt"]

    result = store.get(namespace, "triage_respond")
    if result is None:
        store.put(
            namespace,
            "triage_respond",
            {
                "prompt": prompt_instructions["triage_rules"]["respond"]
            }
        )
        respond_prompt = prompt_instructions["triage_rules"]["respond"]
    else:
        respond_prompt = result.value["prompt"]

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],  #
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        # 从长期记忆当中进行检索，从外部更新准则



        triage_no=ignore_prompt,
        triage_notify=notify_prompt,
        triage_email=respond_prompt,



        examples=examples
    )

    user_prompt = triage_user_prompt.format(
        author=author,
        to=to,
        subject=subject,
        email_thread=email_thread
    )

    result = llm_router.invoke(
        # llm_router是带有分类输出的llm，将llm的输出格式化为route
        [
            ("system", system_prompt),
            ("user", user_prompt)
            # {"role" : "system", "content": system_prompt},
            # {"role": "user", "content": user_prompt}
        ]
    )

    # 处理三种可能的结果
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        # 跳转到响应agent的节点
        update = {
            # 更新状态
            "messages": [
                # {
                #     "role": "user",
                #     "content": f"Respond to the email {state['email_input']}",
                # }
                ("user", f"Respond to email {state['email_input']}")
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        update = None
        goto = END
    elif result.classification == "notify":
        # If real life, this would do something else
        print("🔔 Classification: NOTIFY - This email contains important information")
        update = None
        goto = END
    else:
        raise ValueError(f"Invalid classification: {result.classification}")
    return Command(goto=goto, update=update)

In [4]:
from langchain_core.tools import tool


@tool  # 暂不实现任何逻辑
def write_email(to: str, subject: str, content: str) -> str:
    """Writer and send an email."""
    # Placeholder response - in real app would send email
    # 需要完成发送邮箱的邮箱地址，邮箱主题和邮箱内容
    return f"Email sent to {to} with subject '{subject}'"


@tool
def schedule_meeting(
        attendees: list[str],
        subject: str,
        duration_minutes: int,
        preferred_day: str  # 日历会议
) -> str:
    """Schedule a calendar meeting."""
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"


# 创建记忆管理工具
from langmem import create_manage_memory_tool, create_search_memory_tool

manage_memory_tool = create_manage_memory_tool(
    namespace=(
        "email_assistant",  # 命名空间值
        "{langgraph_user_id}",
        "collection"
    )
)

search_memory_tool = create_search_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)
# 添加新的系统提示词
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""
from langchain.agents.middleware import dynamic_prompt
from langgraph.config import get_config


@dynamic_prompt
def create_prompt(request):
    store = request.runtime.store
    langgraph_user_id = get_config()['configurable']['langgraph_user_id']
    namespace = (langgraph_user_id, )
    result = store.get(namespace, "agent_instructions")
    if result is None:
        # 首次拿到当前用户，那么存储进入
        store.put(
            namespace,
            "agent_instructions",
            {
                "prompt": prompt_instructions["agent_instructions"]
            }
        )
        prompt = prompt_instructions["agent_instructions"]
    else :
        prompt = result.value['prompt']


    # 只返回字符串
    return agent_system_prompt_memory.format(
        instructions=prompt,
        **profile
    )


# 创建react代理函数
from langchain.agents import create_agent

tools = [
    write_email,
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]

response_agent = create_agent(
    "deepseek-v4-flash",
    tools=tools,
    middleware=[create_prompt],
    store=store
)

In [5]:
email_agent = StateGraph(State)
email_agent = email_agent.add_node(triage_router)
email_agent = email_agent.add_node("response_agent", response_agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)


In [6]:
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

Urgent issue - your service is down. Is there a reason why""",
}

In [7]:
config = {"configurable": {"langgraph_user_id": "lance"}}

In [8]:
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

📧 Classification: RESPOND - This email requires a response


In [9]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Respond to email {'author': 'Alice Jones <alice.jones@bar.com>', 'to': 'John Doe <john.doe@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': 'Hi John,\n\nUrgent issue - your service is down. Is there a reason why'}
================================== Ai Message ==================================

I'll start by checking what I already know about this sender and any relevant context.
Tool Calls:
  search_memory (call_00_x5CUNGBjsV7aqRLfZHnm6442)
 Call ID: call_00_x5CUNGBjsV7aqRLfZHnm6442
  Args:
    query: Alice Jones bar.com
  search_memory (call_01_qSkuGSlLYjdk1eKlXhHk8393)
 Call ID: call_01_qSkuGSlLYjdk1eKlXhHk8393
  Args:
    query: service outage incident status API documentation
  search_memory (call_02_lOv4KUmjaOMmVCjwafh62360)
 Call ID: call_02_lOv4KUmjaOMmVCjwafh62360
  Args:
    query: jiarui email identity John Doe company.com
================================= Too

In [10]:
# 查看现有的长期记忆
store.get(("lance",), "agent_instructions").value["prompt"]

'Use these tools when appropriate to help manage emails.'

In [11]:
# 查看现有的长期记忆
store.get(("lance",), "triage_respond").value["prompt"]

'Direct questions from team members, meeting requests, critical bug reports'

In [12]:
# 查看现有的长期记忆
store.get(("lance",), "triage_ignore").value["prompt"]

'Marketing newsletters, spam emails, mass company announcements'

In [13]:
from langmem import create_multi_prompt_optimizer # 让模型根据用户反馈更新记忆

In [14]:
conversations = [
    (
        response['messages'],
        "Always signs your emails `John Doe`"
    )
]

In [15]:
# 定义可能需要更新的提示词列表（每条 prompt 都从长期记忆 store 中实时读取当前值）
prompts = [
    {
        "name": "main_agent",
        "prompt": store.get(("lance",), "agent_instructions").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",  # 更新指导说明：保持指令简短扼要
        "when_to_update": "Update this prompt whenever there is feedback on how the agent should write emails or schedule events",
    },
    {
        "name": "triage-ignore",
        "prompt": store.get(("lance",), "triage_ignore").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be ignored",
    },
    {
        "name": "triage-notify",
        "prompt": store.get(("lance",), "triage_notify").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails the user should be notified of",
    },
    {
        "name": "triage-respond",
        "prompt": store.get(("lance",), "triage_respond").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
    },
]


In [26]:
# 创建多提示优化器
optimizer =create_multi_prompt_optimizer(
    # 改用deepseek-v4-flash作为提示词优化器
    llm,
    # extra_body={"thinking": {"type": "disabled"}},
    kind="prompt_memory",
)

In [27]:
# 提示优化器本身为一个图，因此需要传入轨迹来调用
updated = optimizer.invoke(
    {"trajectories": conversations, "prompts": prompts}
)

In [28]:
print(updated)

[{'name': 'main_agent', 'prompt': 'Use these tools when appropriate to help manage emails. Always sign your emails "John Doe".', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on how the agent should write emails or schedule events'}, {'name': 'triage-ignore', 'prompt': 'Marketing newsletters, spam emails, mass company announcements', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on which emails should be ignored'}, {'name': 'triage-notify', 'prompt': 'Team member out sick, build system notifications, project status updates', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on which emails the user should be notified of'}, {'name': 'triage-respond', 'prompt': 'Direct questions from team members, meeting requests, critical bu

In [29]:
# 更新长期记忆
for i, updated_prompt in enumerate(updated):
    old_prompt = prompts[i];
    if updated_prompt["prompt"] != old_prompt["prompt"]:
        # 如果旧的prompt和当前优化后的prompt不同，那么就执行更新操作
        name = old_prompt["name"]
        print(f"updated {name}") # 打印日志
        if name == "main_agent":
            store.put(
                ("lance",),
                "agent_instructions",
                {"prompt": updated_prompt["prompt"]}
            )
        else :
            raise ValueError

updated main_agent


In [30]:
store.get(("lance",), "agent_instructions").value["prompt"]

'Use these tools when appropriate to help manage emails. Always sign your emails "John Doe".'

In [31]:
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

📧 Classification: RESPOND - This email requires a response


In [33]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Respond to email {'author': 'Alice Jones <alice.jones@bar.com>', 'to': 'John Doe <john.doe@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': 'Hi John,\n\nUrgent issue - your service is down. Is there a reason why'}
================================== Ai Message ==================================
Tool Calls:
  search_memory (call_00_Rg8hYdIbNPPN3J9kMMpZ9279)
 Call ID: call_00_Rg8hYdIbNPPN3J9kMMpZ9279
  Args:
    query: Alice Jones bar.com contact context
  search_memory (call_01_QGcL9O9flMnlhNUclnbn1373)
 Call ID: call_01_QGcL9O9flMnlhNUclnbn1373
  Args:
    query: service down outage API documentation
================================= Tool Message =================================
Name: search_memory

[]
================================= Tool Message =================================
Name: search_memory

[]
================================== Ai Message ====================

In [34]:
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

Urgent issue - your service is down. Is there a reason why""",
}

In [35]:
response = email_agent.invoke(
    {"email_input": email_input},
    config=config
)

📧 Classification: RESPOND - This email requires a response


In [36]:
conversations = [
    (
        response["messages"],
        "Ignore any emails from Alice Jones"
    )
]

In [37]:
# 定义可能需要更新的提示词列表（每条 prompt 都从长期记忆 store 中实时读取当前值）
prompts = [
    {
        "name": "main_agent",
        "prompt": store.get(("lance",), "agent_instructions").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",  # 更新指导说明：保持指令简短扼要
        "when_to_update": "Update this prompt whenever there is feedback on how the agent should write emails or schedule events",
    },
    {
        "name": "triage-ignore",
        "prompt": store.get(("lance",), "triage_ignore").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be ignored",
    },
    {
        "name": "triage-notify",
        "prompt": store.get(("lance",), "triage_notify").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails the user should be notified of",
    },
    {
        "name": "triage-respond",
        "prompt": store.get(("lance",), "triage_respond").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
    },
]

In [38]:
updated = optimizer.invoke(
    {"trajectories": conversations, "prompts": prompts}
)

In [53]:
# 更新长期记忆
for i, updated_prompt in enumerate(updated):
    old_prompt = prompts[i]
    if updated_prompt["prompt"] != old_prompt["prompt"]:
        # 如果旧的prompt和当前优化后的prompt不同，那么就执行更新操作
        name = old_prompt["name"]
        print(f"updated {name}") # 打印日志
        if name == "main_agent":
            store.put(
                ("lance",),
                "agent_instructions",
                {"prompt": updated_prompt["prompt"]}
            )
        elif name == "triage_ignore":
            store.put(
                ("lance",),
                "triage_ignore",
                {"prompt": updated_prompt["prompt"]}
            )
        # else:
        #     raise ValueError

updated main_agent
updated triage-ignore
updated triage-respond


In [54]:
response = email_agent.invoke(
    {"email_input": email_input},
    config = config
)

🚫 Classification: IGNORE - This email can be safely ignored


In [55]:
store.get(("lance",), "triage_ignore").value["prompt"]

'You are an email assistant that reads and replies to incoming emails on behalf of the user. Respond helpfully and concisely; escalate urgent issues appropriately. Skip and do not reply to marketing newsletters, spam emails, and mass company announcements. Ignore emails from senders the user has asked to ignore (e.g., Alice Jones <alice.jones@bar.com>).'